In [27]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from scipy.stats import poisson, skellam
from scipy.optimize import minimize
import datetime


In [28]:
# Data reading
def read_data(path):
    """To read a csv file and return a pandas df"""
    df = pd.read_csv(path)
    return df


In [29]:
from dotenv import load_dotenv
import os

load_dotenv()

sp_2526_path = os.getenv("SP_2025_2026")
sp_2627_path = os.getenv("SP_2026_2027")


In [30]:
sp1 = read_data(sp_2526_path)   # season 25/26 
sp2 = read_data(sp_2627_path)   # season 26/27 


In [31]:

cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']

sp1 = sp1[cols].rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
sp2 = sp2[cols].rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})


data = pd.concat([sp1, sp2], ignore_index=True)

# parser (football-data.co.uk usa dd/mm/yy o dd/mm/yyyy)
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
data = data.dropna(subset=['HomeGoals', 'AwayGoals']).sort_values('Date').reset_index(drop=True)

data.tail()


,Date,HomeTeam,AwayTeam,HomeGoals,AwayGoals
416,2026-09-06,Malaga,Levante,0,0
417,2026-09-06,Valencia,Barcelona,0,5
418,2026-09-06,Espanol,Sevilla,1,1
419,2026-09-07,Getafe,Celta,1,1
420,2026-09-07,Elche,Sociedad,2,3


In [32]:
XI = 0.0018  # decay

reference_date = pd.Timestamp.now().normalize()  # "today": reference for predicting the next round of matches
data['time_diff'] = (reference_date - data['Date']).dt.days

data[['Date', 'HomeTeam', 'AwayTeam', 'HomeGoals', 'AwayGoals', 'time_diff']].tail()


,Date,HomeTeam,AwayTeam,HomeGoals,AwayGoals,time_diff
416,2026-09-06,Malaga,Levante,0,0,5
417,2026-09-06,Valencia,Barcelona,0,5,5
418,2026-09-06,Espanol,Sevilla,1,1,5
419,2026-09-07,Getafe,Celta,1,1,4
420,2026-09-07,Elche,Sociedad,2,3,4


In [33]:
def dc_decay(xi, t):
    return np.exp(-xi * t)


def rho_correction(x, y, lambda_x, mu_y, rho):
    """Dixon-Coles adjustment (tau) for low-scoring matches."""
    if x == 0 and y == 0:
        return 1 - (lambda_x * mu_y * rho)
    elif x == 0 and y == 1:
        return 1 + (lambda_x * rho)
    elif x == 1 and y == 0:
        return 1 + (mu_y * rho)
    elif x == 1 and y == 1:
        return 1 - rho
    else:
        return 1.0


def dc_log_like(x, y, alpha_x, beta_x, alpha_y, beta_y, rho, gamma):
    """Log-likelihood of a result (x home goals, y away goals). 

    alpha = attack strength, beta = defense strength (more negative = better defense),
    gamma = home advantage, rho = correlation for low scores
    """
    lambda_x = np.exp(alpha_x + beta_y + gamma)   # expected goals for the home team
    mu_y = np.exp(alpha_y + beta_x)               # expected goals for the away team
    tau = max(rho_correction(x, y, lambda_x, mu_y, rho), 1e-10)
    return (np.log(tau) +
            np.log(max(poisson.pmf(x, lambda_x), 1e-10)) +
            np.log(max(poisson.pmf(y, mu_y), 1e-10)))


In [34]:
def solve_parameters(dataset, xi=0.0018, init_vals=None,
                      options={'disp': False, 'maxiter': 200}, **kwargs):
    team_names = np.sort(list(set(dataset['HomeTeam'].unique()) | set(dataset['AwayTeam'].unique())))
    n_teams = len(team_names)

    if init_vals is None:
        init_vals = np.concatenate((
            np.random.uniform(0, 1, n_teams),    # initial attack
            np.random.uniform(0, -1, n_teams),   # initial defense
            np.array([0.0, 0.3])                 # rho, home advantage (gamma)
        ))

    def neg_log_likelihood(params):
        attack = dict(zip(team_names, params[:n_teams]))
        defence = dict(zip(team_names, params[n_teams:2 * n_teams]))
        rho, gamma = params[-2:]
        log_like = [
            dc_log_like(
                row.HomeGoals, row.AwayGoals,
                attack[row.HomeTeam], defence[row.HomeTeam],
                attack[row.AwayTeam], defence[row.AwayTeam],
                rho, gamma
            ) * dc_decay(xi, row.time_diff)
            for row in dataset.itertuples()
        ]
        return -sum(log_like)

    constraints = [{'type': 'eq', 'fun': lambda x, n=n_teams: sum(x[:n]) - n}]
    opt_output = minimize(neg_log_likelihood, init_vals, options=options,
                           constraints=constraints, **kwargs)

    if not opt_output.success:
        print("Warning: optimization did not fully converge ->", opt_output.message)

    param_names = (["attack_" + t for t in team_names] +
                   ["defence_" + t for t in team_names] +
                   ['rho', 'home_adv'])
    return dict(zip(param_names, opt_output.x))


In [35]:
params = solve_parameters(data, xi=XI)

print("rho (correlation of low markers):", round(params['rho'], 4))
print("home_adv (home-field advantage):", round(params['home_adv'], 4))


rho (correlation of low markers): -0.0215
home_adv (home-field advantage): 0.3475


In [36]:
# MATCH PREDICTIONS
def calc_means(param_dict, home_team, away_team):
    home_goals_avg = np.exp(param_dict['attack_' + home_team] + param_dict['defence_' + away_team]
                             + param_dict['home_adv'])
    away_goals_avg = np.exp(param_dict['attack_' + away_team] + param_dict['defence_' + home_team])
    return home_goals_avg, away_goals_avg


def dc_simulate_match(param_dict, home_team, away_team, max_goals=10):
    home_avg, away_avg = calc_means(param_dict, home_team, away_team)

    home_probs = [poisson.pmf(i, home_avg) for i in range(max_goals + 1)]
    away_probs = [poisson.pmf(i, away_avg) for i in range(max_goals + 1)]

    match_matrix = np.outer(np.array(home_probs), np.array(away_probs))

    correction = np.array([[rho_correction(hg, ag, home_avg, away_avg, param_dict['rho'])
                             for ag in range(2)] for hg in range(2)])
    match_matrix[:2, :2] = match_matrix[:2, :2] * correction

    return match_matrix


In [37]:
def score_probability(param_dict, home_team, away_team, home_goals, away_goals):
    """Probabilidad (en %) de un marcador exacto."""
    match_matrix = dc_simulate_match(
        param_dict, home_team, away_team,
        max_goals=max(home_goals, away_goals, 6)
    )
    return match_matrix[home_goals][away_goals] * 100


## Example

Replace `'Barcelona'` and `'Levante'` with the teams you want to predict
(they must match the names used in the CSV).


In [40]:
home_team, away_team = 'Barcelona', 'Levante'

matrix = dc_simulate_match(params, home_team, away_team, max_goals=8)

home_win = np.sum(np.tril(matrix, -1))
draw = np.sum(np.diag(matrix))
away_win = np.sum(np.triu(matrix, 1))

print(f"{home_team} Win: {home_win:.1%}")
print(f"Draw: {draw:.1%}")
print(f"{away_team} Win: {away_win:.1%}")


Barcelona Win: 87.2%
Draw: 7.9%
Levante Win: 3.7%


In [ ]:
# Probability of an exact score
prob = score_probability(params, home_team, away_team, 5, 0)
print(f"P({home_team} 5 - 0 {away_team}) = {prob:.2f}%")


P(Barcelona 5 - 0 Levante) = 6.71%
